# 01 · Query building and data collection (Google Trends, MediaCloud)

This notebook builds the queries and collects every series used in the paper.

| Section | Output | Used in |
|---|---|---|
| 2. Google Trends snowball | `data/processed/queries/gt_snowball_terms_<code>.csv`, `gt_snowball_rounds_<code>.csv` | query choice, Figure 9 |
| 3. MediaCloud k-grams | `data/processed/queries/mc_stories_<code>.csv`, `mc_kgrams_<code>.csv` | query choice, Figure 10 |
| 4. Monthly series 2015–2025 | `data/processed/monthly/GT_monthly.csv`, `MC_monthly.csv` | Figure 1 |
| 5. Daily seasonal windows | `GT_<code>_Granger.csv`, `MC_<code>_Granger_normalized.csv` | notebooks 02–03 |

**You do not need to run this notebook to reproduce the analysis.** The daily windows
used in the paper are already in `data/processed/daily/`. Run it to refresh the data,
change a query or add a campaign.

**Before running**
* Google Trends is accessed through `pytrends`, an unofficial client. Google limits
  it heavily (HTTP 429), so every call waits 20–60 s; the snowball alone can take
  30–60 minutes per campaign.
* MediaCloud needs a free API key from <https://search.mediacloud.org>. Set it in the
  shell **before** starting Jupyter: `export MEDIACLOUD_API_KEY="..."`. Never write the
  key in the notebook.
* Each section runs only if its switch in `RUN` below is set to `True`.
* Google Trends values are computed from a sample of searches, so a new download gives
  slightly different numbers from the files used in the paper.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "phc").exists())
sys.path.insert(0, str(ROOT / "src"))

import datetime as dt
import pandas as pd

from phc.config import (CAMPAIGNS, ORDER, DAILY_DIR, DATA_PROCESSED, MONTHLY_DIR,
                        QUERY_DIR, MONTHLY_TIMEFRAME, gt_query, mc_query)
from phc import collect as C

RUN = {
    "snowball": False,       # section 2  (Google Trends, slow)
    "kgrams": False,         # section 3  (MediaCloud + article download)
    "monthly": False,        # section 4  (Google Trends + MediaCloud)
    "daily": False,          # section 5  (Google Trends + MediaCloud)
}
CODES = ORDER                # campaigns to process, e.g. ["MB"] for Mars Bleu only

# New daily files are written here; copy them to data/processed/daily/ to use them
# in notebooks 02-03 (the files used in the paper are not overwritten).
DAILY_OUT = DATA_PROCESSED / "daily_recollected"
DAILY_OUT.mkdir(parents=True, exist_ok=True)

## 1. Queries

The current queries are defined once, in `src/phc/config.py`, and shown below. After
sections 2–3, edit `gt_terms` and `mc_terms` there with the terms you validated.

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.DataFrame({"Google Trends": {CAMPAIGNS[c].label: gt_query(CAMPAIGNS[c]) for c in CODES},
              "MediaCloud": {CAMPAIGNS[c].label: mc_query(CAMPAIGNS[c]) for c in CODES}})

## 2. Google Trends query: snowball sampling

Starting from the seeds of each campaign, the related queries returned by Google Trends
are collected, then the related queries of the best new terms, and so on for three
rounds. Terms matching the exclusion patterns (e.g. "recette", "cocktail") are dropped.

The output table is **reviewed by hand**: add a column `keep` (True/False) in the CSV,
then copy the kept terms into `gt_terms` in `config.py`. The `relevance` column (TF-IDF
similarity to the seeds) only helps to sort the candidates.

The stability check then downloads the signal of the cumulative query after each round
(Figure 9 of the paper, drawn in notebook 02): when the curve stops changing shape,
extra terms add volume but no new dynamics.

In [ ]:
if RUN["snowball"]:
    gt = C.gt_client()
    for code in CODES:
        camp = CAMPAIGNS[code]
        print(f"== {camp.label}")
        terms = C.snowball(camp, MONTHLY_TIMEFRAME, rounds=3, client=gt)
        terms.to_csv(QUERY_DIR / f"gt_snowball_terms_{code}.csv", index=False)
        rounds = C.snowball_series(camp, terms, MONTHLY_TIMEFRAME, per_round=2, client=gt)
        rounds.to_csv(QUERY_DIR / f"gt_snowball_rounds_{code}.csv", index=False)
        print(f"   {len(terms)} terms; queries by round:")
        print(rounds.groupby("round")["query"].first().to_string())
else:
    print("Section skipped (RUN['snowball'] is False).")

## 3. MediaCloud query: k-grams of article texts

A first set of articles is retrieved with a short query (the campaign name and its
disease or behaviour). Their full texts are downloaded with `newspaper4k` (the maintained
fork of `newspaper3k`; same `Article` interface), cleaned of boilerplate lines (cookie
banners, newsletter links…) and stop words, and the most frequent bigrams and trigrams
are counted. Relevant k-grams are then selected by hand and copied into `mc_terms`.

To keep the run short, articles are sampled from the three most recent campaign windows.

In [ ]:
SEED_QUERIES = {
    "OR": '"octobre rose" OR "cancer du sein"',
    "Mov": '"movember" OR "cancer de la prostate"',
    "MB": '"mars bleu" OR "cancer colorectal"',
    "DJ": '"dry january" OR "janvier sans alcool"',
}
MAX_STORIES_PER_SEASON = 300

if RUN["kgrams"]:
    mc = C.mc_client()
    for code in CODES:
        camp = CAMPAIGNS[code]
        print(f"== {camp.label}")
        seasons = list(camp.season_windows())[-3:]
        stories = pd.concat([
            C.mc_stories(mc, SEED_QUERIES[code], start, end, max_stories=MAX_STORIES_PER_SEASON)
             .assign(season=label)
            for label, start, end in seasons], ignore_index=True)
        stories = stories.drop_duplicates("url")
        print(f"   {len(stories)} articles; downloading texts…")
        stories["text"] = [C.fetch_text(u) for u in stories["url"]]
        stories = stories[stories["text"].str.len() > 200]
        stories.to_csv(QUERY_DIR / f"mc_stories_{code}.csv", index=False)
        kg = C.kgram_counts(stories["text"].tolist())
        kg.to_csv(QUERY_DIR / f"mc_kgrams_{code}.csv", index=False)
        print(f"   {len(stories)} texts kept. Top bigrams / trigrams:")
        for n in (2, 3):
            print("  ", ", ".join(kg[kg["n"] == n].head(10)["kgram"]))
else:
    print("Section skipped (RUN['kgrams'] is False).")

## 4. Monthly series, 2015–2025

* **Google Trends:** one request over 2015–2025 (longer than five years, so Google
  returns monthly values; 100 = maximum month).
* **MediaCloud:** daily article counts summed by month. The number of articles in the
  whole collection (`total_count`) is kept too, so that counts can be divided by the
  size of the collection, which changes over time.

In [ ]:
if RUN["monthly"]:
    gt = C.gt_client()
    mc = C.mc_client()
    gt_rows, mc_rows = [], []
    for code in CODES:
        camp = CAMPAIGNS[code]
        print(f"== {camp.label}")
        s = C.gt_interest(gt, gt_query(camp), MONTHLY_TIMEFRAME)
        gt_rows.append(pd.DataFrame({"code": code, "date": s.index, "value": s.values}))
        d = C.mc_counts(mc, mc_query(camp), dt.date(2015, 1, 1), dt.date(2025, 12, 31))
        m = d.resample("MS").sum()
        mc_rows.append(pd.DataFrame({"code": code, "date": m.index, "value": m["count"].values,
                                     "total_count": m["total_count"].values}))
    pd.concat(gt_rows).to_csv(MONTHLY_DIR / "GT_monthly.csv", index=False)
    pd.concat(mc_rows).to_csv(MONTHLY_DIR / "MC_monthly.csv", index=False)
    print("Saved", (MONTHLY_DIR / "GT_monthly.csv").relative_to(ROOT),
          "and", (MONTHLY_DIR / "MC_monthly.csv").relative_to(ROOT))
else:
    print("Section skipped (RUN['monthly'] is False).")

## 5. Daily seasonal windows

For each campaign and each year, the window around the campaign (see `config.py`) is
short enough for Google Trends to return **daily** values, with 100 = the maximum day of
that season. MediaCloud daily article counts over the same window are divided by their
seasonal maximum and multiplied by 100, so both series are on the same 0–100 scale
within each season. This is exactly how the files in `data/processed/daily/` were built.

Raw MediaCloud counts are saved as well (`MC_<code>_daily_counts.csv`).

In [ ]:
def collect_daily(code, gt, mc, out_dir=DAILY_OUT):
    camp = CAMPAIGNS[code]
    q_gt, q_mc = gt_query(camp), mc_query(camp)
    gt_parts, mc_parts = [], []
    for season, start, end in camp.season_windows():
        print(f"   {season}: {start} → {end}")
        s = C.gt_interest(gt, q_gt, f"{start} {end}")
        gt_parts.append(pd.DataFrame({
            "date": s.index, "campaign_index": s.values, "campaign": camp.label,
            "query": q_gt, "season": season, "window_start": start, "window_end": end}))
        d = C.mc_counts(mc, q_mc, start, end)
        peak = d["count"].max()
        mc_parts.append(pd.DataFrame({
            "date": d.index, "n_articles": 100 * d["count"] / peak if peak > 0 else 0.0,
            "count": d["count"].values, "total_count": d["total_count"].values,
            "season": season}))
    gt_df, mc_df = pd.concat(gt_parts), pd.concat(mc_parts)
    gt_df.to_csv(out_dir / f"GT_{code}_Granger.csv", index=False)
    mc_df[["date", "n_articles"]].to_csv(out_dir / f"MC_{code}_Granger_normalized.csv", index=False)
    mc_df.to_csv(out_dir / f"MC_{code}_daily_counts.csv", index=False)


if RUN["daily"]:
    gt, mc = C.gt_client(), C.mc_client()
    for code in CODES:
        print(f"== {CAMPAIGNS[code].label}")
        collect_daily(code, gt, mc)
    print("Saved in", DAILY_OUT.relative_to(ROOT))
else:
    print("Section skipped (RUN['daily'] is False).")

## 6. Check the new files

A quick comparison of recollected daily GT values with the files used in the paper
(correlation by campaign). A correlation well below 1 usually means a different query,
or Google's sampling noise for low-volume terms.

In [ ]:
for code in CODES:
    new = DAILY_OUT / f"GT_{code}_Granger.csv"
    if not new.exists():
        continue
    a = pd.read_csv(DAILY_DIR / f"GT_{code}_Granger.csv", parse_dates=["date"], encoding="utf-8-sig")
    b = pd.read_csv(new, parse_dates=["date"])
    j = a.merge(b, on="date", suffixes=("_paper", "_new"))
    print(f"{CAMPAIGNS[code].label:13s} {len(j):5d} common days, "
          f"r = {j['campaign_index_paper'].corr(j['campaign_index_new']):.3f}")